In [11]:
import duckdb
con = duckdb.connect()
con.sql("CREATE OR REPLACE TABLE players AS SELECT * FROM 'data/cookie_cats.csv'")

In [31]:
con.sql("""
  SELECT version, COUNT(*) AS n_players
  FROM players
  GROUP BY version
  """)

┌─────────┬───────────┐
│ version │ n_players │
│ varchar │   int64   │
├─────────┼───────────┤
│ gate_40 │     45489 │
│ gate_30 │     44700 │
└─────────┴───────────┘

In [32]:
con.sql("""SELECT userid AS user FROM players WHERE version = 'gate_40' AND sum_gamerounds <2 """)

┌────────────┐
│    user    │
│   int64    │
├────────────┤
│        483 │
│       2502 │
│       4076 │
│       4902 │
│       6156 │
│       7509 │
│       8801 │
│       9828 │
│       9849 │
│      10276 │
│        ·   │
│        ·   │
│        ·   │
│    9969675 │
│    9970608 │
│    9979148 │
│    9981794 │
│    9983936 │
│    9988600 │
│    9988650 │
│    9989991 │
│    9994647 │
│    9995246 │
└────────────┘
  4846 rows 
  (20 shown) 

In [33]:
con.sql("""SELECT version, COUNT(userid) FROM players GROUP by version
""") #How many players are in each version?

┌─────────┬───────────────┐
│ version │ count(userid) │
│ varchar │     int64     │
├─────────┼───────────────┤
│ gate_40 │         45489 │
│ gate_30 │         44700 │
└─────────┴───────────────┘

In [27]:
#Are there duplicate userids?
con.sql("""
  SELECT userid, COUNT(*) AS times_seen
  FROM players
  GROUP BY userid
  HAVING COUNT(*) > 1
  """)

┌────────┬────────────┐
│ userid │ times_seen │
│ int64  │   int64    │
└────────┴────────────┘
        0 rows       

In [30]:
con.sql("""
SELECT version, COUNT(*) FROM players
WHERE sum_gamerounds =0
GROUP BY version""")

┌─────────┬──────────────┐
│ version │ count_star() │
│ varchar │    int64     │
├─────────┼──────────────┤
│ gate_30 │         1937 │
│ gate_40 │         2057 │
└─────────┴──────────────┘

In [44]:
con.sql("""SELECT version, AVG(sum_gamerounds) AS avg_rounds, MEDIAN(sum_gamerounds) AS median_rounds, MAX(sum_gamerounds) AS max_round FROM players
GROUP BY version
""")

┌─────────┬───────────────────┬───────────────┬───────────┐
│ version │    avg_rounds     │ median_rounds │ max_round │
│ varchar │      double       │    double     │   int64   │
├─────────┼───────────────────┼───────────────┼───────────┤
│ gate_40 │ 51.29877552814966 │          16.0 │      2640 │
│ gate_30 │ 52.45626398210291 │          17.0 │     49854 │
└─────────┴───────────────────┴───────────────┴───────────┘

In [45]:
con.sql("""SELECT version, AVG(CAST(retention_1 AS INTEGER)) AS retention_1day, AVG(CAST(retention_7 AS INTEGER)) AS retention_7day FROM players GROUP BY version
""")

┌─────────┬─────────────────────┬─────────────────────┐
│ version │   retention_1day    │   retention_7day    │
│ varchar │       double        │       double        │
├─────────┼─────────────────────┼─────────────────────┤
│ gate_30 │  0.4481879194630872 │ 0.19020134228187918 │
│ gate_40 │ 0.44228274967574577 │ 0.18200004396667327 │
└─────────┴─────────────────────┴─────────────────────┘

In [ ]:
# Two important observations:
#  1. The average is about 3× the median. That means the data is right-skewed: most players play a little, and a few play a lot. That's why the median is the better "typical player" number here. It's also why you'll later use the
#     Mann-Whitney U test for engagement rather than a t-test.
#  2. 49,854 rounds in 14 days is not realistic. That's about 3,560 rounds a day, or roughly 150 an hour around the clock. It's almost certainly a data error or a bot. The next-highest player (gate_40's 2,640) is far below it.

In [46]:
#identify the player
con.sql("""
  SELECT *
  FROM players
  WHERE sum_gamerounds = 49854
  """)

┌─────────┬─────────┬────────────────┬─────────────┬─────────────┐
│ userid  │ version │ sum_gamerounds │ retention_1 │ retention_7 │
│  int64  │ varchar │     int64      │   boolean   │   boolean   │
├─────────┼─────────┼────────────────┼─────────────┼─────────────┤
│ 6390605 │ gate_30 │          49854 │ false       │ true        │
└─────────┴─────────┴────────────────┴─────────────┴─────────────┘

In [48]:
 con.sql("""
  CREATE OR REPLACE TABLE players_clean AS
  SELECT *
  FROM players
  WHERE sum_gamerounds < 49854
  """)

In [49]:
 con.sql("""
  SELECT version, COUNT(*) AS n_players, MAX(sum_gamerounds) AS max_rounds
  FROM players_clean
  GROUP BY version
  """)

┌─────────┬───────────┬────────────┐
│ version │ n_players │ max_rounds │
│ varchar │   int64   │   int64    │
├─────────┼───────────┼────────────┤
│ gate_30 │     44699 │       2961 │
│ gate_40 │     45489 │       2640 │
└─────────┴───────────┴────────────┘

In [50]:
#FROM players_clean
con.sql("""SELECT version, AVG(sum_gamerounds) AS avg_rounds, MEDIAN(sum_gamerounds) AS median_rounds, MAX(sum_gamerounds) AS max_round 
FROM players_clean
GROUP BY version
""")

┌─────────┬────────────────────┬───────────────┬───────────┐
│ version │     avg_rounds     │ median_rounds │ max_round │
│ varchar │       double       │    double     │   int64   │
├─────────┼────────────────────┼───────────────┼───────────┤
│ gate_30 │ 51.342110561757536 │          17.0 │      2961 │
│ gate_40 │  51.29877552814966 │          16.0 │      2640 │
└─────────┴────────────────────┴───────────────┴───────────┘

In [51]:
con.sql("""SELECT version, AVG(CAST(retention_1 AS INTEGER)) AS retention_1day, AVG(CAST(retention_7 AS INTEGER)) AS retention_7day 
FROM players_clean
GROUP BY version
""")

┌─────────┬─────────────────────┬─────────────────────┐
│ version │   retention_1day    │   retention_7day    │
│ varchar │       double        │       double        │
├─────────┼─────────────────────┼─────────────────────┤
│ gate_30 │  0.4481979462627799 │ 0.19018322557551623 │
│ gate_40 │ 0.44228274967574577 │ 0.18200004396667327 │
└─────────┴─────────────────────┴─────────────────────┘